# Assignment 2: DeepFake Image Detection
### Course: Foundations of Data Science (FDS)
**Topic**: Deep Learning approach for Deepfake Image Detection  
**Reference Paper**: *MesoNet: a Compact Facial Video Forgery Detection Network*, Darius Afchar, Vincent Nozick, Junichi Yamagishi, Isao Echizen (IEEE WIFS 2018)  
**Methodology**: Mesoscopic Convolutional Neural Network (Meso4) utilizing shallow layers to extract micro-texture artifacts rather than high-level semantic features.

---
## 1. Environment Setup & Configuration
This notebook covers dataset loading from Google Drive, implementation of the MesoNet architecture, Cross-validation, and accuracy reporting.

In [ ]:
# Check environment and configure paths
import os
import sys

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    print("[*] Running in Google Colab environment.")
    from google.colab import drive
    drive.mount('/content/drive')
    !pip install -q matplotlib seaborn scikit-learn opencv-python
else:
    print("[*] Running in Local environment.")
    
print("Environment setup complete!")

## 2. Dataset Loading and Preprocessing
Loading the images directly from Google Drive and analyzing the dataset distribution.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("[*] Loading Dataset from Google Drive...")

# ---------------------------------------------------------
# Your 'real' and 'fake' folders are directly in your main Drive folder!
base_path = '/content/drive/MyDrive'
# ---------------------------------------------------------

real_dir = os.path.join(base_path, 'real')
fake_dir = os.path.join(base_path, 'fake')

X = []
y = []
max_images_per_class = 1500 # Adjust if you want to load more/less

# Load Fake Images (Label 0)
if os.path.exists(fake_dir):
    print(f"Loading fake images from {fake_dir}...")
    for img_name in os.listdir(fake_dir)[:max_images_per_class]: 
        img_path = os.path.join(fake_dir, img_name)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (128, 128))
            X.append(img / 255.0)
            y.append(0)
else:
    print(f"[!] Path not found: {fake_dir}")

# Load Real Images (Label 1)
if os.path.exists(real_dir):
    print(f"Loading real images from {real_dir}...")
    for img_name in os.listdir(real_dir)[:max_images_per_class]:
        img_path = os.path.join(real_dir, img_name)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (128, 128))
            X.append(img / 255.0)
            y.append(1)
else:
    print(f"[!] Path not found: {real_dir}")

X = np.array(X)
y = np.array(y)

print(f"\n[*] Total Images Loaded: {len(X)}")

if len(X) == 0:
    print("WARNING: No images found! Please double-check your 'base_path' above.")
else:
    unique, counts = np.unique(y, return_counts=True)
    print(f"Class distribution: {dict(zip(['Fake', 'Real'], counts))}")

    # Plot Data Analysis Screenshot
    plt.figure(figsize=(8, 5))
    sns.barplot(x=['Fake (0)', 'Real (1)'], y=counts, palette='viridis')
    plt.title('Exploratory Data Analysis: Class Distribution')
    plt.ylabel('Number of Images')
    plt.show()

    # Show samples
    fig, axes = plt.subplots(1, 4, figsize=(15, 4))
    for i in range(min(4, len(X))):
        axes[i].imshow(X[i])
        axes[i].set_title(f"Label: {'Real' if y[i]==1 else 'Fake'}")
        axes[i].axis('off')
    plt.suptitle("Visual Comparison: Authentic vs Synthetic Deepfakes")
    plt.show()

## 3. Architecture - Implementing MesoNet (Meso4)
We implement the Meso4 architecture proposed in the paper, which uses shallow convolutions to capture mesoscopic image features.

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, BatchNormalization, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

def create_meso4_model(input_shape=(128, 128, 3)):
    x = Input(shape=input_shape)
    
    x1 = Conv2D(8, (3, 3), padding='same', activation='relu')(x)
    x1 = BatchNormalization()(x1)
    x1 = MaxPooling2D(pool_size=(2, 2), padding='same')(x1)
    
    x2 = Conv2D(8, (5, 5), padding='same', activation='relu')(x1)
    x2 = BatchNormalization()(x2)
    x2 = MaxPooling2D(pool_size=(2, 2), padding='same')(x2)
    
    x3 = Conv2D(16, (5, 5), padding='same', activation='relu')(x2)
    x3 = BatchNormalization()(x3)
    x3 = MaxPooling2D(pool_size=(2, 2), padding='same')(x3)
    
    x4 = Conv2D(16, (5, 5), padding='same', activation='relu')(x3)
    x4 = BatchNormalization()(x4)
    x4 = MaxPooling2D(pool_size=(4, 4), padding='same')(x4)
    
    y = Flatten()(x4)
    y = Dropout(0.5)(y)
    y = Dense(16)(y)
    y = Dropout(0.5)(y)
    y = Dense(1, activation='sigmoid')(y)
    
    model = Model(inputs=x, outputs=y, name="Meso4_Deepfake_Detector")
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

if len(X) > 0:
    meso4 = create_meso4_model()
    meso4.summary()

## 4. Cross-Validation and Training
To ensure generalizability, we evaluate the model using Stratified K-Fold Cross-Validation.

In [ ]:
from sklearn.model_selection import StratifiedKFold

if len(X) > 0:
    n_splits = 3 # 3-fold cross-validation
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    fold_no = 1
    acc_per_fold = []
    loss_per_fold = []
    model = None # Keep the last model for final evaluation

    for train_index, val_index in skf.split(X, y):
        print(f'\n--- Training for Fold {fold_no} ---')
        
        X_train, X_val = X[train_index], X[val_index]
        y_train, y_val = y[train_index], y[val_index]
        
        model = create_meso4_model()
        history = model.fit(X_train, y_train, epochs=8, batch_size=32, validation_data=(X_val, y_val), verbose=1)
        
        scores = model.evaluate(X_val, y_val, verbose=0)
        print(f'[*] Score for Fold {fold_no}: Loss = {scores[0]:.4f}; Accuracy = {scores[1]*100:.2f}%')
        acc_per_fold.append(scores[1] * 100)
        loss_per_fold.append(scores[0])
        
        fold_no += 1

    print(f'\n[*] Average Accuracy across all folds: {np.mean(acc_per_fold):.2f}%')

## 5. Accuracy Report & Comprehensive Benchmarking
Generating the final visualizations including the confusion matrix and the classification report.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

if len(X) > 0 and model is not None:
    y_pred_probs = model.predict(X_val)
    y_pred = (y_pred_probs > 0.5).astype(int).flatten()

    # Confusion Matrix Visualization
    cm = confusion_matrix(y_val, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
    plt.title('Out-of-fold Confusion Matrix for MesoNet Ensemble')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

    # Classification Report
    print("\n================ Detailed Accuracy Report ================")
    print(classification_report(y_val, y_pred, target_names=['Fake', 'Real']))
    print("==========================================================")